# 03 - Modelling

Stage 3: time-aware split, two models, PR-AUC selection.

Why not a random split? Every behavioural feature here is cumulative. A random split lets the model see later transactions from a destination account during training and earlier ones at test time, which inflates results and says nothing about tomorrow.


In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

from src import config, preprocessing, data_profiling, feature_engineering
from src.utils import setup_logging
setup_logging()

# Use --nrows style subsetting here to keep the notebook responsive.
# PaySim is time-ordered, so this is an EARLY TIME WINDOW, not a random sample.
df = preprocessing.load_paysim(nrows=1_000_000)
df = preprocessing.prepare(df)
df.head()

from src import train_models, evaluate_models
featured, feature_columns = feature_engineering.build_features(df)
featured = feature_engineering.fill_missing(featured, feature_columns)

In [ ]:
split = train_models.time_aware_split(featured)
masks = split['masks']
import pandas as pd
pd.DataFrame({k: v for k, v in split['summary'].items() if isinstance(v, dict)}).T

In [ ]:
y = featured[config.TARGET].to_numpy()
X = featured[feature_columns]
sets = {name: (X[m], y[m.to_numpy()]) for name, m in masks.items()}

models = train_models.build_models(fast=True)
fitted = train_models.fit_models(models, *sets['train'])

In [ ]:
val_scores = {n: train_models.predict_proba(m, sets['validation'][0]) for n, m in fitted.items()}
evaluate_models.compare_models(sets['validation'][1], val_scores)

Selection is on validation PR-AUC. The test period is scored once, at the end.


In [ ]:
best = 'random_forest' if 'random_forest' in fitted else list(fitted)[0]
test_scores = train_models.predict_proba(fitted[best], sets['test'][0])
evaluate_models.evaluate(sets['test'][1], test_scores)

In [ ]:
pd.DataFrame(evaluate_models.top_k_capture(sets['test'][1], test_scores))